# GPROF-IR

In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

In [ ]:
from pansat.time import TimeRange
from pansat.utils import resample_data_binned

In [ ]:
from pansat.utils import make_latlon_area
lon_min = -125
lat_min = 24
lon_max = -65
lat_max = 51
imerg_grid = make_latlon_area(
    lon_min,
    lat_min,
    lon_max,
    lat_max,
    (lon_max - lon_min) / 0.1, 
    (lat_max - lat_min) / 0.1
)
imerg_grid

In [ ]:
config = "gmi_3"

In [ ]:
list(gprof_ir_files.values())[0]

In [ ]:
from datetime import datetime
from pansat.products.satellite.persiann import ccs_1h

def get_date(path: Path) -> datetime:
    return datetime.strptime(path.name.split("_")[-1][:-3], "%Y%m%d%H")
    
gprof_ir_files = sorted(list(Path(f"/gdata2/simon/gprof_ir/results/gprof_ir_{config}").glob("gprof_ir*.nc")))
gprof_ir_files = {
    get_date(path): path for path in gprof_ir_files
}
print(len(gprof_ir_files))

In [ ]:
from datetime import datetime
from gprof_ir.imerg import load_imerg_data
from pansat.products.satellite import gpm
from pansat.utils import resample_data_binned


def extract_gprof_ir_data(year, month, day, hour) -> xr.Dataset:
    """
    Extract GPROF-IR data over CONUS

    Args:
        year: The year
        month: the month
        day: the day
        hout the hour

    Return:
        The GPROF-IR precip rate.
    """
    imerg_data = []
    time = datetime(year, month, day, hour)
    path = gprof_ir_files[time]
    with xr.open_dataset(path) as data:
        data = data[["surface_precip"]].compute()
    data = resample_data_binned(data, imerg_grid)
    return data.mean("time")

In [ ]:
output_path = Path(f"/gdata2/simon/gprof_ir/conus/gprof_ir_{config}")
output_path.mkdir(parents=True, exist_ok=True)

In [ ]:
start_time = np.datetime64("2022-07-01")
end_time = np.datetime64("2023-01-01")

for hour in np.arange(start_time, end_time, np.timedelta64(1, "h")):
    date = hour.astype("datetime64[s]").item()
    print(date)
    output_file = output_path / date.strftime("gprof_ir_%Y%m%d%H%M%S.nc")
    try:
        gprof_ir_data = extract_gprof_ir_data(date.year, date.month, date.day, date.hour)
        for var in ["surface_precip"]:
            gprof_ir_data[var].encoding = {
                "zlib": True,
                "dtype": "float32"
            }
        gprof_ir_data.to_netcdf(output_file)
    except Exception as exc:
        raise exc